In [2]:
import requests
import pandas as pd
import numpy as np
import datetime
from astropy.time import Time
from LINZ.CORS_Timeseries import TimeseriesList, CoordApiTimeseries

In [3]:
# Define the timeframe
start = datetime.datetime(2025, 4, 1)
end = datetime.datetime(2025, 4, 30)

In [5]:
# Define station codes, unhashtag the CORS wanted --- don't forget to change the name of the CSV output file at the end

#Trimble V
#no issues/good to go
codes=['VAK2',	'VALE',	'VCHR',	'VHAM',	'VHN2',	'VHRA',	'VLEE',	'VMAY',	'VNPE',	'VOMA',	'VOTG',	'VOTO',	'VPOA', 'VPRK',	'VPUK',	'VTEA',	'VTEK',	
'VTEM', 'VWAN', 'VWKU', 'VWPK', 'VZQN','VOMA','VMKA','VSHF','VTES','VTOK','VWK2','VWM2','VPN2']
#needs further investigation, try a couple of time periods
#First Round internally consistent jan/feb/mar
#codes=[]

#Position Partners PP
#no issues, but ppsp needs to be a lower order than 3, 4?
#codes =['PPCW','PPNL','PPOR','PPPK','PPRS','PPSP']
#Going with Feb Coords
#codes=['PPK2']
#Inactive codes=['PPMK']

# PositioNZ CORS
#codes=['AUCK','BLUF','CHTI','CORM','DNVK','DUND','GISB','GLDB','HAAS','HAMT','HAST','HIKB','HOKI','KAIK','KTIA','LEXA','LKTA','MAHO','MAVL','METH','MQZG','MTJO','NLSN','NPLY','PYGR','TAUP','TRNG','VGMT','WAIM','WANG','WARK','WEST','WGTN','WHKT','WHNG','WRPA']

# Elliot Sinclair CORS
#codes=['ESBR','ESGM','ESKV','ESNL','ESPM','ESQT','ESRN','ESTR']
#Inactive codes=['ESCM']

#Global Survey
#no issues/good to go
#codes=['GSAA','GSAB','GSAM','GSAY','GSBA','GSBD','GSCC','GSCT','GSDA','GSDR','GSGR','GSHT','GSHU','GSHW','GSIN','GSKA','GSKK','GSLI','GSLW',
#'GSLX','GSMG','GSOM','GSOK','GSPE','GSPL','GSPM','GSPN','GSPO','GSPR','GSPU','GSQT','GSRA','GSRL','GSRO','GSSB','GSTA','GSTH','GSTK','GSTU',
#'GSTW','GSUH','GSUT','GSWI','GSWR','GSWU','GSNP','GSHO','GSGL','GSBN','GSBR','GSMA','GSMD','GSMO','GSPK','GSWA','GSMH','GSTI','GSWH']
#need further investigation
#first round good going with Jan 2025 coords, issues with coords identified in spreadsheet


#Synergy SY
#no issues good to go
#codes=['SYAS','SYCC','SYCM','SYDF','SYHA','SYHW','SYM2','SYMV','SYNL','SYQN','SYT2','SYTA','SYTS','SYWM','SYET','SYHA','SYBM','SYRD','SYLN','SYWK','SYPP']
#Needs further investigation, syns may have recently stopped

#Survey Hire SH
#no issues good to go
#codes=['SHAR','SHMD','SHPA','SHML','SHSA']
#very intermittent, and changable coords, wonder whether its worth even updating
#codes=['SHML','SHSA']

count = len(codes)
print(f'number of cors: {count} \n')

# Calculate important parameters
crd_epoch = end + (start - end) / 2
print(f'crd_epoch: {crd_epoch} --- nominal coordinate epoch of the output coordinates\n')

astropy_time_object = Time(crd_epoch, format='datetime')
crd_epoch_decimal_year = astropy_time_object.decimalyear
print(f'crd_epoch_decimal_year : {crd_epoch_decimal_year} --- as above but in decimal year format used in Online Coordinate Converter conversions\n')

ndays = (end - start).days
print(f'ndays: {ndays} --- the number of days from start date to end date\n')
print(f'ndays/2: {int(ndays/2)} --- every mark must have at least this many days of data\n')
print(f'ndays/8: {int(ndays/8)} --- every mark must have at least this many days of data before and after the crd_epoch\n')

# Initialise DataFrame to store model ITRF coordinates
df_model_ITRF_crds = pd.DataFrame()

# Calculate daily averages
daily_avg_list = []
for code in codes:
    try:
        print(f"Processing station: {code}")
        ts = CoordApiTimeseries(code, solutiontype='d20f_52_code_B', after=start, before=end)  # To Download PositioNZ CORS data, set solutiontype='d20f_52_code_A', to download private CORS data, set solutiontype='d20f_52_code_B'
        dates, xyz = ts.withoutOutliers().getObs(enu=False)  # Remove outliers

        # Create DataFrame from the observations
        df_obs = pd.DataFrame(xyz, columns=['x', 'y', 'z'], index=dates)
        
        # Calculate daily averages
        daily_avg = df_obs.resample('D').mean()
        daily_avg['date'] = daily_avg.index.strftime('%Y-%m-%d')
        daily_avg['station'] = code
        
        daily_avg_list.append(daily_avg)
    except Exception as e:
        print(f"Failed to process station {code} with error: {e}")

# Combine all daily averages into a single DataFrame and sort by GNSS station code in alphabetical order
combined_daily_avg = pd.concat(daily_avg_list).sort_index()

# Display the first 5 rows of the combined DataFrame 
print(combined_daily_avg.head())


number of cors: 30 

crd_epoch: 2025-04-15 12:00:00 --- nominal coordinate epoch of the output coordinates

crd_epoch_decimal_year : 2025.286301369863 --- as above but in decimal year format used in Online Coordinate Converter conversions

ndays: 29 --- the number of days from start date to end date

ndays/2: 14 --- every mark must have at least this many days of data

ndays/8: 3 --- every mark must have at least this many days of data before and after the crd_epoch

Processing station: VAK2
Processing station: VALE
Processing station: VCHR
Processing station: VHAM
Processing station: VHN2
Processing station: VHRA
Failed to process station VHRA with error: No d20f_52_code_B timeseries data found for VHRA after 2025-04-01 and before 2025-04-30
Processing station: VLEE
Failed to process station VLEE with error: No d20f_52_code_B timeseries data found for VLEE after 2025-04-01 and before 2025-04-30
Processing station: VMAY
Processing station: VNPE
Processing station: VOMA
Processing stati

In [56]:
# Assuming combined_daily_avg is already defined from the previous cell

# Some of the feilds in the ITRF download may be NaN, infinity or extremely large or small values, these are not compatable with the NZTM/NZV2016 conversion.
# Identify and print NaN values
nan_values = combined_daily_avg[combined_daily_avg.isna().any(axis=1)]
print("NaN values:")
print(nan_values)

# Identify and print Infinity values
inf_values = combined_daily_avg[(combined_daily_avg == np.inf) | (combined_daily_avg == -np.inf)].dropna(how='all')
print("Infinity values:")
print(inf_values)

# Filter out rows with NaN, Infinity, or extremely large/small float values
filtered_combined_daily_avg = combined_daily_avg.dropna(subset=['x', 'y', 'z'])
filtered_combined_daily_avg = filtered_combined_daily_avg[
    (filtered_combined_daily_avg['x'] != np.inf) & (filtered_combined_daily_avg['x'] != -np.inf) &
    (filtered_combined_daily_avg['y'] != np.inf) & (filtered_combined_daily_avg['y'] != -np.inf) &
    (filtered_combined_daily_avg['z'] != np.inf) & (filtered_combined_daily_avg['z'] != -np.inf)
]

# Function to convert coordinates to NZTM2000 and elevation to NZVD2016 using LINZ API
def convert_coordinates(input_crds, crd_epoch_decimal_year):
    occ_api = "https://www.geodesy.linz.govt.nz/api/conversions/v1/convert-to"
    coordlist = {
        "crs": "LINZ:ITRF2020_XYZ",  # Correct input parameter
        "coordinateOrder": ["geocentricX", "geocentricY", "geocentricZ"],
        "coordinateEpoch": crd_epoch_decimal_year,
        "coordinates": input_crds
    }
    params = {"crs": "LINZ:NZTM/NZVD2016"} # Conversions to other curcuits are easy to do by changing the LINZ coordinate code here. 

    response = requests.post(occ_api, params=params, json=coordlist)
    if response.status_code == 200:
        converted = response.json()
        return converted['coordinateList']['coordinates']
    else:
        print(f"Error converting coordinates: {response.text}")
        return []

# Prepare coordinates for conversion
input_crds = filtered_combined_daily_avg[['x', 'y', 'z']].values.tolist()

# Convert coordinates to NZTM2000 and elevation to NZVD2016
converted_coords = convert_coordinates(input_crds, crd_epoch_decimal_year)

# Add converted coordinates to the DataFrame
if converted_coords:
    filtered_combined_daily_avg[['nztm2000_lon', 'nztm2000_lat', 'nzvd2016_elev']] = pd.DataFrame(converted_coords, index=filtered_combined_daily_avg.index)

# Display the updated DataFrame to ensure the new coordinate system matches the expected output
print(filtered_combined_daily_avg.head())


NaN values:
             x   y   z        date station
2025-01-23 NaN NaN NaN  2025-01-23    SHSA
2025-01-24 NaN NaN NaN  2025-01-24    SHSA
2025-01-25 NaN NaN NaN  2025-01-25    SHSA
2025-01-26 NaN NaN NaN  2025-01-26    SHSA
2025-01-27 NaN NaN NaN  2025-01-27    SHMD
2025-01-27 NaN NaN NaN  2025-01-27    SHML
2025-01-27 NaN NaN NaN  2025-01-27    SHSA
2025-01-27 NaN NaN NaN  2025-01-27    SHAR
2025-01-28 NaN NaN NaN  2025-01-28    SHMD
2025-01-28 NaN NaN NaN  2025-01-28    SHSA
2025-01-28 NaN NaN NaN  2025-01-28    SHAR
2025-01-28 NaN NaN NaN  2025-01-28    SHML
2025-01-29 NaN NaN NaN  2025-01-29    SHAR
2025-01-29 NaN NaN NaN  2025-01-29    SHMD
2025-01-29 NaN NaN NaN  2025-01-29    SHML
2025-01-29 NaN NaN NaN  2025-01-29    SHSA
2025-01-30 NaN NaN NaN  2025-01-30    SHML
2025-01-30 NaN NaN NaN  2025-01-30    SHMD
2025-01-30 NaN NaN NaN  2025-01-30    SHSA
2025-01-30 NaN NaN NaN  2025-01-30    SHAR
2025-01-31 NaN NaN NaN  2025-01-31    SHML
2025-01-31 NaN NaN NaN  2025-01-31    SHSA

In [57]:
# Save the updated DataFrame with the converted coordinates to a CSV file
filtered_combined_daily_avg.to_csv('Survey_Hire_converted_coordinates.csv', index=True) # Change the output CSV file name and location to match the data and location being downloaded

# Print a success message
print("The updated DataFrame with the converted coordinates has been saved")


The updated DataFrame with the converted coordinates has been saved
